# NeuroDecode–LaBraM | Phase 4: pretrained partial/full fine-tuning

**Scientific status: development experiment.** This notebook compares **probe, partial (last 2 blocks), full** on **Train and Validation only**. The 15-person held-out Test partition is **not** used for training, evaluation, or model selection.

The 104-participant PhysioNet EEGMMIDB dataset must already be attached under **Kaggle → Add Input** or present in the same active Kaggle Session. **No EDF/HDF5 download or rebuild is attempted.** The notebook checks its exact SHA-256 before model training.

Each candidate trains from the *same pinned pretrained LaBraM revision*. Model checkpoints and EEG files are excluded from the shareable ZIP; best-epoch checkpoints remain in private Kaggle working output for version saving. Successful completion alone does **not** establish a Phase 4 PASS until the scientific evidence is reviewed.

In [ ]:
from pathlib import Path
import sys, os, json, hashlib, subprocess
from datetime import datetime, timezone

REPO_URL = "https://github.com/mohamad679/labram-mi-project.git"
REPO_SHA = "15468a4392faa8019ed3f3ac5746314756851514"  # Phase 4 validated runner/test snapshot
RUNNER_SHA256 = "c1dadc4dc5bc0a3b92ff20d973be1b271861cf0f123cf7959be55c8fc408623d"
DATA_SHA256 = "f6d2c7824f6aaa522286ea8ee5253b0220602117fc69d7b49c3af6e6a1324f7a"
PRETRAINED_REVISION = "0563b6c626e7b40d9a36653b763715db94d945d7"
CACHE_OVERRIDE = ""  # Optional absolute path to existing physionet_mi_104subjects.h5
ROOT = Path('/kaggle/working' if Path('/kaggle').exists() else '/mnt/data')
CODE_DIR = ROOT / 'neurodecode_phase4_repo'
RESULT_ROOT = ROOT / 'NeuroDecode_Phase4'
MODEL_RESULTS = RESULT_ROOT / 'models'
ZIP_PATH = ROOT / 'NeuroDecode_Phase4_Validation_Results.zip'

def sha256_file(path):
    d = hashlib.sha256()
    with Path(path).open('rb') as f:
        for block in iter(lambda: f.read(8*1024*1024), b''):
            d.update(block)
    return d.hexdigest()
print('Notebook protocol: validation-only, pinned code revision, no test inference')
print('Python:', sys.version.split()[0]); print('Working root:', ROOT)


In [ ]:
# Install the versions already validated in the previous Kaggle run.
# pyriemann 0.10 is incompatible with moabb 1.7.2: use pyriemann 0.12.
# Do not downgrade/reinstall the GPU-provided torch build.
packages = [
    "braindecode[hub]==1.5.1", "moabb==1.7.2", "mne==1.13.2",
    "pyriemann==0.12", "h5py>=3.10,<5", "scikit-learn>=1.6,<2",
    "numpy>=2,<3", "pytest>=8,<9", "PyYAML>=6,<7",
]
subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', *packages], check=True)
import torch, braindecode, moabb, mne, numpy, sklearn, h5py
assert torch.cuda.is_available(), 'CUDA is required; set Kaggle Accelerator = GPU before running.'
VERSIONS = {
  'python':sys.version.split()[0], 'torch':torch.__version__,
  'braindecode':braindecode.__version__, 'moabb':moabb.__version__,
  'mne':mne.__version__, 'numpy':numpy.__version__,
  'sklearn':sklearn.__version__, 'h5py':h5py.__version__,
  'device':torch.cuda.get_device_name(0),
}
print(json.dumps(VERSIONS,indent=2))


In [ ]:
# Checkout exact Phase 4 training code. Fail if its provenance has drifted.
if not CODE_DIR.exists():
    subprocess.run(['git','clone','--quiet',REPO_URL,str(CODE_DIR)],check=True)
subprocess.run(['git','-C',str(CODE_DIR),'checkout','--quiet','--detach',REPO_SHA],check=True)
head = subprocess.check_output(['git','-C',str(CODE_DIR),'rev-parse','HEAD'],text=True).strip()
assert head == REPO_SHA, f'Unexpected git HEAD {head}'
runner = CODE_DIR/'scripts/run_phase4_finetune.py'
actual = sha256_file(runner)
assert actual == RUNNER_SHA256, f'Unexpected runner source checksum: {actual}'
print('Pinned repository:',head)
print('Runner SHA256:',actual)


In [ ]:
# Run local negative-control and frozen-parameter tests before any training.
subprocess.run([
    sys.executable, '-m','pytest','-q',
    'tests/test_phase4_trainability.py',
    'tests/test_phase4_validation.py',
],cwd=CODE_DIR,check=True)
print('All Phase 4 offline scientific contract tests passed')


In [ ]:
# Find ONLY an existing, previously validated schema-3 EEG cache.
if CACHE_OVERRIDE:
    candidates = [Path(CACHE_OVERRIDE)]
else:
    candidates = []
    for directory in [Path('/kaggle/input'), Path('/kaggle/working')]:
        if directory.is_dir():
            candidates.extend(directory.rglob('physionet_mi_104subjects.h5'))
    candidates = sorted(set(candidates),key=str)
print('Found', len(candidates), 'candidate HDF5 files')
for path in candidates: print(str(path), round(path.stat().st_size / 2**30,3),'GiB')
valid = []
for path in candidates:
    value = sha256_file(path)
    print('SHA256',path, value)
    if value == DATA_SHA256: valid.append(path)
if len(valid) != 1:
    raise RuntimeError('Exactly one matching Phase-2 cache is required. Attach the saved 104-subject notebook output; do not rebuild data.')
CACHE = valid[0]
print('PASSED: exact 104-participant HDF5 SHA256', DATA_SHA256)


## Execute three controlled training regimes

Predeclared settings: **15 epochs maximum**, patience **5**, microbatch **16**, gradient accumulation **2**, head LR **1×10⁻³**, partial backbone LR **1×10⁻⁴**, full backbone LR **3×10⁻⁵**. Train-only class weighting and best-validation-macro-F1 selection. Frozen model parameters must be byte-identical before and after optimization.

The three regimes load the same official immutable pretrained LaBraM weights; training can be computationally demanding. Training logs are streamed to this cell and persisted for scientific review. The system refuses to overwrite an existing run directory.

In [ ]:
# Avoid Run All if you already generated MODEL_RESULTS in this session.
if MODEL_RESULTS.exists():
    raise FileExistsError('Existing Phase-4 results detected: preserve them and use a fresh Kaggle session / output location')
RESULT_ROOT.mkdir(parents=True, exist_ok=True)
log = RESULT_ROOT / 'phase4_execution.log'
command = [
    sys.executable,'-u','-m','scripts.run_phase4_finetune',
    '--cache',str(CACHE),'--split-manifest','configs/phase3_locked_split.json',
    '--output-dir',str(MODEL_RESULTS),
    '--strategies','probe','partial','full',
    '--seed','42','--epochs','15','--patience','5',
    '--batch-size','16','--accumulate','2',
    '--head-lr','0.001','--partial-lr','0.0001','--full-lr','0.00003',
    '--weight-decay','0.0001','--grad-clip','1.0',
    '--last-n-blocks','2','--bootstrap','250','--device','cuda'
]
print('Executing validated Phase 4 code; no Test evaluation is performed.',flush=True)
with log.open('w',encoding='utf-8') as stream:
    with subprocess.Popen(command,cwd=CODE_DIR,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
                          text=True,bufsize=1) as proc:
        for line in proc.stdout:
            print(line,end='',flush=True)
            stream.write(line)
            stream.flush()
        rc = proc.wait()
if rc:
    raise RuntimeError(f'Phase4 runner exit {rc}: inspect {log}; do NOT claim PASS')
assert (MODEL_RESULTS/'phase4_summary.json').is_file()
print('Training completed and all checkpoint reload/frozen-weight gates passed')


In [ ]:
# Summary and verification of output-only metadata, not held-out predictions.
summary = json.loads((MODEL_RESULTS/'phase4_summary.json').read_text())
assert summary['status'] == 'VALIDATION_COMPLETE_NOT_A_HELDOUT_RESULT'
assert summary['dataset_sha256'] == DATA_SHA256
assert summary['source_code_sha256'] == RUNNER_SHA256
assert summary['strategies_completed'] == ['probe','partial','full']
print('MODEL / VALIDATION MACRO-F1 / BALANCED ACC / BEST EPOCH')
for strategy in summary['strategies_completed']:
    report = json.loads((MODEL_RESULTS/strategy/'report.json').read_text())
    assert report['checkpoint_reloaded'] and report['validation_only'] and not report['test_predictions_computed']
    assert report['validation_epochs'] == 2436
    assert report['frozen_parameters_unchanged']
    print(strategy, report['validation_metrics']['macro_f1'],
          report['validation_metrics']['balanced_accuracy'], report['best_epoch'])
print('All three 2,436-epoch validation-only result bundles are present')


In [ ]:
# Export *only* small scientific reports, code/env hashes, safe NPZ predictions, and log.
# Checkpoint binaries (~20+ MB) stay in Kaggle working output and are NOT exported.
import zipfile
if ZIP_PATH.exists():
    raise FileExistsError(f'Refusing to overwrite {ZIP_PATH}')
metadata = {
    'source_commit':REPO_SHA,'runner_sha256':RUNNER_SHA256,
    'dataset_sha256':DATA_SHA256,'pretrained_revision':PRETRAINED_REVISION,
    'python_and_package_versions':VERSIONS,'scope':'Validation ONLY; no held-out Test',
    'results_zip_excludes_checkpoints_and_raw_EEG':True,
    'generated_at_utc':datetime.now(timezone.utc).isoformat(),
}
(RESULT_ROOT/'execution_provenance.json').write_text(json.dumps(metadata,indent=2)+'\n')
with zipfile.ZipFile(ZIP_PATH,'x',compression=zipfile.ZIP_DEFLATED) as z:
    z.write(RESULT_ROOT/'execution_provenance.json',arcname='execution_provenance.json')
    z.write(RESULT_ROOT/'phase4_execution.log',arcname='phase4_execution.log')
    for p in MODEL_RESULTS.rglob('*'):
        if not p.is_file(): continue
        if p.suffix.lower() in ('.json','.npz'):
            z.write(p,arcname=str(Path('models')/p.relative_to(MODEL_RESULTS)))
print('VALIDATION-ONLY ZIP READY:',ZIP_PATH)
print('ZIP SIZE MB:',round(ZIP_PATH.stat().st_size/1e6,2))
print('ZIP SHA256:',sha256_file(ZIP_PATH))
print('Preserve checkpoint binaries via Kaggle Save Version outputs. Do not upload EEG raw data.')


## Scientific handling

1. Download/share **`NeuroDecode_Phase4_Validation_Results.zip`** for review. No model quality is claimed before independent inspection of all validation predictions and gradients.
2. Save a Kaggle output version if you need reproducible selected model binaries. These weight files are **not** inside the ZIP.
3. Do **not** modify or evaluate the locked Test cohort; do not infer publication-level generalization from this single development Validation seed.
4. If the notebook fails (including CUDA OOM), share the small **log/traceback**. Do not silently lower the batch size or alter learning rates and mix results with this preregistered run.